# Value Learning

参考王树森《深度强化学习》一书。

下面是一张图，描述了整个知识体系应该是怎样的。

![value learnig](assets/value_learning_arch.png)

## 1. 环境准备

这里来个最简单的 `CartPole-v1`.

In [1]:
import gymnasium as gym

env = gym.make("CartPole-v1")

print(env)
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

<TimeLimit<OrderEnforcing<PassiveEnvChecker<CartPoleEnv<CartPole-v1>>>>>
Observation space: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
Action space: Discrete(2)


In [4]:
state, info = env.reset()

print("state =", state)
print("shape =", state.shape)
print("info =", info)

state = [0.0198882  0.02319125 0.03686673 0.01567904]
shape = (4,)
info = {}


在这个系统中，有两个物理对象，一个小车，一个杆。状态向量有四维，分别对应小车位置、小车速度、杆的角度与杆的角速度。在这个状态编码下系统具有一阶马尔科夫性。而 action space 只有两个离散的值：`0` 和 `1`，分别对应向左与向右。下面体验一下，以尽快熟悉开发环境。

In [6]:
action = 0

next_state, reward, terminated, truncated, info = env.step(action)

print(f"next_state: {state}")
print(f"reward: {reward}")
print(f"terminated: {terminated}")
print(f"truncated: {truncated}")

env.close()

next_state: [0.0198882  0.02319125 0.03686673 0.01567904]
reward: 1.0
terminated: False
truncated: False


注意，这里和旧书中的 `observation, reward, done, info = env.step(action)` 不一样。

接下来看看一个随机 agent. （懒得录屏了，后面自己把代码运行一下就看得到）

In [14]:
env = gym.make("CartPole-v1", render_mode="human")
state, info = env.reset()
total_reward = 0

for t in range(1000):
    action = env.action_space.sample()
    state, reward, terminated, truncated, info = env.step(action)
    total_reward += reward

    if terminated or truncated:
        break

print(total_reward)
env.close()

14.0


大致理解了整个流程，尝试一个简单的启发式 Agent:

In [17]:
env = gym.make("CartPole-v1", render_mode="human")
state, info = env.reset()
total_reward = 0

for t in range(1000):
    pole_angle = state[2]
    action = 0 if pole_angle < 0 else 1
    state, reward, terminated, truncated, info = env.step(action)
    total_reward += reward

    if terminated or truncated:
        break

print(total_reward)
env.close()

52.0


玩到这里，开始进入正题了

## 2. DQN

我们需要训练一个神经网络 $$ Q_\theta: \mathbb{R}^4 \to \mathbb{R}^2 $$，其根据当前 state 判断每个动作的价值。

下面先来创建 DQN 模型。

In [18]:
import random
from collections import deque, namedtuple

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

In [19]:
class DQN(nn.Module):
    def __init__(self, state_dim, action_dim, latent_dim=10):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(state_dim, latent_dim),
            nn.ReLU(),
            nn.Linear(latent_dim, action_dim),
        )

    def forward(self, x):
        return self.net(x)